[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Multiomics-Analytics-Group/course_multi-omics_analysis/blob/main/notebooks/05_Visualising_Networks/03_nx.ipynb)

# Multi-omics Data Science

# Introduction to NetworkX (Network Analysis)

NetworkX is a Python library for building and analysing networks of any kind.

It provides a framework for:
- Creating networks
- Analysing networks with graph algorithms
- Visualisation
- Interfacing with other network software

## Objectives

- Creating a network
- Handling nodes and edges
- Node and edge attributes
- Visualisation
- Building a network from real omics data (protein co-abundance)
- Community detection and interactive visualisation

## Setup

We install the two extra libraries used later in the notebook
(`python-louvain`, imported as `community`, for community detection and `pyvis`
for interactive network views) and define where the course data lives.

In [ ]:
# Extra libraries used in the second half of the notebook.
# On Colab (or a fresh environment) run this once; it is a no-op if they are already installed.
%pip install -q pyvis python-louvain

In [ ]:
import os

# Where the course data lives. The notebook works both inside a clone of the
# course repository and standalone (e.g. on Colab), where files are read from GitHub.
COURSE_REPO = "Multiomics-Analytics-Group/course_multi-omics_analysis"
BRANCH = "main"
BASE_URL = f"https://raw.githubusercontent.com/{COURSE_REPO}/{BRANCH}"


def course_file(relative_path):
    """Path to a course data file: a local copy if we are in the repo, otherwise the raw GitHub URL."""
    for prefix in ("", "..", "../..", "../../.."):
        candidate = os.path.join(prefix, relative_path) if prefix else relative_path
        if os.path.exists(candidate):
            return candidate
    return f"{BASE_URL}/{relative_path}"


PROTEOMICS_MATRIX = course_file("proteomics/data/protein_groups_matrix.tsv")
SAMPLE_METADATA = course_file("metadata/sample_metadata.tsv")
print(PROTEOMICS_MATRIX)

## Creating a Network

We create an empty network (no nodes).

In [26]:
import networkx as nx

In [27]:
G = nx.Graph()

### Adding Nodes

**A single node**

In [28]:
G.add_node(1)

**Several nodes at once (from a list)**

In [29]:
G.add_nodes_from([2, 3])

**Number of nodes**

In [30]:
G.number_of_nodes()

3

### Adding Edges

**A single edge**

In [ ]:
G.add_edge(1, 2)

**Several edges at once (from a list)**

In [32]:
G.add_edges_from([(1, 2),(1, 3)])

**Number of edges**

In [33]:
G.number_of_edges()

2

**List of adjacent nodes (neighbours)**

In [34]:
list(G.neighbors(1))

[2, 3]

**Number of adjacent nodes (degree)**

In [35]:
list(nx.degree(G))

[(1, 2), (2, 1), (3, 1)]

## Node and Edge Attributes

### Node attributes

In [ ]:
G.add_node(1, name='ALB')

In [ ]:
G.add_node(2, name='HP')

**Show the attributes**

In [ ]:
list(G.nodes(data=True))

### Edge attributes

In [ ]:
G.add_edge(1, 2, weight=5.7, color='blue')

In [ ]:
G.add_edges_from([(3, 4), (4, 5)], color='red', weight=2.)

In [ ]:
G.add_edges_from([(3, 1), (1, 5)], color='blue', weight=2.)

**List the edge attributes**

In [ ]:
list(G.edges(data=True))

## Visualisation

In [ ]:
%matplotlib inline

In [ ]:
import matplotlib.pyplot as plt

**Drawing with the default layout**

In [ ]:
nx.draw(G)

**Drawing with labels**

In [ ]:
nx.draw_networkx(G)

**Drawing with a spectral layout**

In [ ]:
nx.draw_spectral(G)

**Drawing with a circular layout**

In [ ]:
nx.draw_circular(G)

## Practice

1. Create a network with 10 nodes (`path_graph`)

2. Connect the nodes so that every node has degree of at least 2 (`degree`)

3. Add `color` as an attribute of nodes 1 to 5

4. Add a weight to the edges between nodes (1,2), (2,3), (3,4)

5. Visualise the network with several layouts

## Loading Real Data

From here on we stop using toy graphs and build a network from the course
dataset: serum proteomics of **45 septic patients**, 15 per group, measured with
diaPASEF and quantified with DIA-NN.

| Group | Samples | Meaning |
| --- | --- | --- |
| `Con` | `Con1`...`Con15` | sepsis with negative cultures |
| `CSKP` | `KP1`...`KP15` | carbapenem-**susceptible** *Klebsiella pneumoniae* sepsis |
| `CRKP` | `CRKP1`...`CRKP15` | carbapenem-**resistant** *K. pneumoniae* sepsis |

We will turn this matrix into a **protein co-abundance network**: nodes are
protein groups, and two proteins are connected when their abundances move
together across the 45 patients. This is the same construction that the
Multi-omics II session (this afternoon) extends to two omics layers at once.

### Reading Data with Pandas

**Comma/Tab separated files (.csv, .tsv)**

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv(PROTEOMICS_MATRIX, sep='\t', header=0)
print(df.shape)
df.head()

### From a Protein Matrix to an Edge List

The file has four annotation columns (`protein_group`, `protein_names`, `genes`,
`description`) followed by one column per sample: the 45 patients plus three
`QC_pool` injections that we drop.

To get an edge list we:

1. keep the 45 patient columns and take `log2` of the intensities,
2. label each row with its gene symbol (falling back to the accession),
3. keep the protein groups that are **quantified in every patient** and, among
   those, the **150 most variable** ones,
4. transpose to *samples x proteins* and correlate the proteins with `.corr()`,
5. keep the pairs with `abs(r) > 0.7`.

> **What does an edge mean here?** It is a *correlation*, not a physical
> interaction. Two proteins whose serum levels rise and fall together across
> patients may be co-regulated, may be released from the same tissue, or may
> simply share an upstream driver such as the severity of the infection. The
> edge is a hypothesis to follow up, not a mechanism.

In [ ]:
ANNOTATION_COLS = ["protein_group", "protein_names", "genes", "description"]

# The 45 patient samples: everything that is not annotation and not a QC pool
samples = [c for c in df.columns
           if c not in ANNOTATION_COLS and not c.startswith("QC_pool")]
print(len(samples), "patient samples:", samples[:3], "...", samples[-3:])

# log2 intensities, one row per protein group, labelled with the gene symbol
# (some rows have no gene symbol, or several; we take the first one, or the accession)
labels = df["genes"].fillna(df["protein_group"]).str.split(";").str[0]

log_data = np.log2(df[samples])
log_data.index = labels.values
log_data = log_data[~log_data.index.duplicated()]
print("log2 matrix:", log_data.shape)
log_data.iloc[:5, :5]

In [ ]:
# 1) keep only protein groups quantified in every patient (no missing values)
complete = log_data[log_data.notna().sum(axis=1) == len(samples)]
print("complete protein groups:", complete.shape[0])

# 2) of those, keep the 150 most variable ones -- constant proteins carry no
#    information about co-abundance
most_variable = complete.var(axis=1).sort_values(ascending=False).index[:150]
matrix = complete.loc[most_variable].T          # samples x proteins
print("matrix for correlation (samples x proteins):", matrix.shape)
matrix.iloc[:5, :5]

In [ ]:
# Pearson correlation between every pair of proteins across the 45 patients
correlation = matrix.corr()

# Keep the upper triangle only, so each pair appears once, and turn it into a long table
upper = np.triu(np.ones(correlation.shape, dtype=bool), k=1)
pairs = correlation.where(upper).stack()
pairs.index.names = ["protein_1", "protein_2"]
pairs = pairs.rename("r").reset_index()

THRESHOLD = 0.7
edges = pairs[pairs["r"].abs() > THRESHOLD].copy()
print(f"{len(edges)} edges with |r| > {THRESHOLD}")
edges.head()

### Converting the Data Frame into a NetworkX Graph

In [ ]:
G = nx.from_pandas_edgelist(edges, "protein_1", "protein_2", edge_attr="r")
print(G)
print("density:", round(nx.density(G), 4),
      "| connected components:", nx.number_connected_components(G))

### Visualising the Graph

In [ ]:
plt.figure(figsize=(11, 9))
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, node_size=180, node_color="#8ecae6", edge_color="#cccccc", with_labels=False)
nx.draw_networkx_labels(G, pos, font_size=7)
plt.title("Protein co-abundance network (|r| > 0.7)")
plt.axis("off")
plt.show()

### Identifying Communities in the Network

Communities (or modules) of a network are subsets of nodes with a higher
density of connections among themselves than with the rest of the network.
Identifying these modules can be interpreted as functional relatedness,
similarity, or classification.

Here we use a well-known community detection method, the **Louvain method**,
widely used to extract communities from large networks. It optimises
*modularity*, which compares the density of edges inside a community with the
density of edges leaving it. Optimising modularity gives, in theory, the best
grouping of nodes into communities.
https://en.wikipedia.org/wiki/Louvain_method

In a co-abundance network a community is a group of proteins that behave as one
block across the patients -- often a complex, a pathway, or proteins released
together from the same tissue.

In [ ]:
import community.community_louvain as cm

communities = cm.best_partition(G)
print(len(set(communities.values())), "communities found")

> **Note.** `community.best_partition` comes from the `python-louvain`
> package, which is an extra dependency. NetworkX itself now ships a Louvain
> implementation, `nx.community.louvain_communities(G, seed=42)`, which returns
> a list of sets instead of a node -> community dictionary. We keep
> `python-louvain` here because that is the API used in the rest of the course
> material, but the built-in function is a dependency-free alternative.

### Turning the Result into a Data Frame

In [ ]:
communities_df = pd.DataFrame.from_dict(communities, orient="index")
communities_df.columns = ["community"]
print(communities_df["community"].value_counts().head())
communities_df.head()

### Picking One Community

In [ ]:
# The largest community is usually the most interesting one to look at
community_id = communities_df["community"].value_counts().idxmax()
community_nodes = communities_df[communities_df["community"] == community_id].index.tolist()
print("community", community_id, "with", len(community_nodes), "proteins")

In [ ]:
community_nodes

### Extracting the Community as a Subgraph

In [ ]:
C = G.subgraph(community_nodes)

### Visualising the Community

In [ ]:
plt.figure(figsize=(8, 7))
nx.draw(C, with_labels=True, node_size=600, node_color="#ffb703",
        edge_color="#999999", font_size=8)
plt.title(f"Community {community_id} of the co-abundance network")
plt.axis("off")
plt.show()

# Other libraries for network visualisation

## PyVis

Interactive network visualisation -- https://pyvis.readthedocs.io/en/latest/index.html

In [ ]:
!pip install pyvis

In [ ]:
from pyvis.network import Network
from IPython.display import display, HTML

In [ ]:
nt = Network('600px', '100%', notebook=True, cdn_resources='in_line')
nt.from_nx(G)
nt.save_graph("protein_coabundance.html")
HTML(filename="protein_coabundance.html")

## Using Attributes

PyVis can use attributes attached to the nodes to change how those nodes look.

The attributes are:

**['size', 'value', 'title', 'x', 'y', 'label', 'color']**

In [ ]:
nx_graph = nx.cycle_graph(6)
nx_graph.nodes[0]['title'] = 'ALB'
nx_graph.nodes[0]['group'] = 4
nx_graph.nodes[1]['title'] = 'HP'
nx_graph.nodes[1]['group'] = 1
nx_graph.nodes[2]['title'] = 'APOA1'
nx_graph.nodes[2]['group'] = 1
nx_graph.nodes[3]['title'] = 'PKM'
nx_graph.nodes[3]['group'] = 2
nx_graph.nodes[4]['title'] = 'ENO1'
nx_graph.nodes[4]['group'] = 2
nx_graph.nodes[5]['title'] = 'TUBB'
nx_graph.nodes[5]['group'] = 3
nt = Network('500px', '500px', notebook=True, cdn_resources='in_line')
# populates the nodes and edges data structures
nt.from_nx(nx_graph)
nt.save_graph("attributes_graph.html")
HTML(filename="attributes_graph.html")

PyVis also lets you add **interactive options** to change some of the
parameters of your network on the fly. It is very useful when you are looking
for the best parameters to display your network.

To switch these options on, use:

``` python
net.show_buttons(filter_=['nodes','edges', 'physics'])
```

In [ ]:
# The same co-abundance network, with the Louvain community as the node "group"
# (PyVis colours nodes by group) and the correlation shown on hover.
P = G.copy()
for node in P.nodes():
    P.nodes[node]['group'] = communities[node]
    P.nodes[node]['title'] = f"{node} (community {communities[node]}, degree {P.degree(node)})"
    P.nodes[node]['label'] = node
for u, v, data in P.edges(data=True):
    data['title'] = f"r = {data['r']:.2f}"

nt = Network('600px', '100%', notebook=True, cdn_resources='in_line')
nt.from_nx(P)
nt.show_buttons(filter_=['nodes', 'edges', 'physics'])
nt.save_graph("coabundance_communities.html")
HTML(filename="coabundance_communities.html")

## Exercise

Build a network yourself and use node attributes to change how it looks.

Suggested route, using the sample metadata:

1. Read the patient groups with
   `metadata = pd.read_csv(SAMPLE_METADATA, sep="\t")` and look at the
   `sample_id` / `group` columns (`Con`, `CSKP`, `CRKP`).
2. Rebuild the co-abundance network **within one group only** (15 samples), for
   example the `CRKP` patients, and compare it with the `Con` network: how many
   edges survive the same `|r| > 0.7` threshold? Are the hub proteins the same?
3. Colour the nodes by Louvain community, and set `size` from the node degree.
4. Remember that with only 15 samples per group a correlation of 0.7 is much
   easier to reach by chance than with 45 -- so a denser per-group network is
   not automatically a more biological one.